In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import json
plt.style.use('../graph_preset.mplstyle')

In [2]:
read_path = Path(".//results.h5")

In [3]:
with h5py.File(read_path, "r") as f: # read_paths[#] that you want to read
    print(f"--- Structure of {read_path} ---")

    def print_structure(name, obj):
        # データセットの場合は形状とデータ型も表示
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} (Dataset) | Shape: {obj.shape}, Dtype: {obj.dtype}")
        # グループの場合はグループ名を表示
        elif isinstance(obj, h5py.Group):
            print(f"  {name} (Group)")

    f.visititems(print_structure)
    print("---------------------------------")

--- Structure of results.h5 ---
  input (Group)
  learning_curve (Group)
  output (Group)
  output/repeat_1 (Dataset) | Shape: (130, 5), Dtype: float32
---------------------------------


In [4]:
df_data = dict()

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")
        df = pd.DataFrame(obj[:])
        df_data[name] = df

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")

        # columns 属性があれば JSON から復元
        cols_attr = obj.attrs.get("columns", None)
        columns = None
        if cols_attr is not None:
            # 古い h5py だと bytes / np.bytes_ で返ることがある
            if isinstance(cols_attr, (bytes, np.bytes_)):
                cols_attr = cols_attr.decode("utf-8")
            columns = json.loads(cols_attr)

        # DataFrame 化（列名があれば使う）
        data = obj[:]  # dset[:, :] と同じ
        if columns is not None:
            df = pd.DataFrame(data, columns=columns)
        else:
            df = pd.DataFrame(data)

        df_data[name] = df


with h5py.File(read_path, "r") as f:
    print(f"--- Loading all datasets from {read_path} ---")
    f.visititems(store_dataset)
    print("---------------------------------------------")

print("\n--- Dictionary Keys ---")
print(list(df_data.keys()))
print("-----------------------")

--- Loading all datasets from results.h5 ---
  Loading: output/repeat_1 | Shape: (130, 5)
---------------------------------------------

--- Dictionary Keys ---
['output/repeat_1']
-----------------------


In [5]:
pd.set_option('display.max_rows', None)

In [6]:
df_data["output/repeat_1"]

,s1x,s1y,S11,Acq,best
0,0.696806,0.741483,4.572438,NaN,4.572438
1,-1.714480,-0.892982,6.178440,NaN,4.572438
2,0.254110,-1.596750,6.151381,NaN,4.572438
3,-0.501485,-1.304809,5.793420,NaN,4.572438
4,0.116252,1.939949,5.214386,NaN,4.572438
5,-1.382773,-0.256902,5.649987,NaN,4.572438
6,-1.552515,0.023632,5.639528,NaN,4.572438
7,-0.859122,0.903035,3.882377,NaN,3.882377
8,-1.273945,0.258158,5.172320,NaN,3.882377
9,-1.977018,1.382445,7.366222,NaN,3.882377
